In [ ]:
!pip install -q anthropic folium beautifulsoup4 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 763.1/763.1 kB 16.0 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
EMAIL_ADDRESS = userdata.get("EMAIL_ADDRESS")

In [ ]:
USER_AGENT = f"CIS3120 Student {EMAIL_ADDRESS}"

REQUEST_HEADERS = {"User-Agent": USER_AGENT}

In [ ]:
import os
import re
import json
import time
import requests
import folium
import pandas as pd
from datetime import date, timedelta
from bs4 import BeautifulSoup
import anthropic

# --- CONFIGURATION ---
client = anthropic.Anthropic()
MODEL_ID = "claude-haiku-4-5-20251001"

EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
NOMINATIM_PAUSE = 1.1  # OSM requires 1 request per second

FINANCIAL_SERVICES_PHRASES = [
    '"new branch"', '"branch opening"', '"branch closure"', '"branch closing"',
    '"branch consolidation"', '"office closure"', '"operations center"',
    '"data center"', '"new location"', '"trading floor"',
    '"wealth management office"', '"private banking center"'
]

TRAVEL_HOSPITALITY_PHRASES = [
    '"new property"', '"new hotel"', '"hotel opening"', '"resort opening"',
    '"property opening"', '"brand conversion"', '"new route"', '"new gateway"',
    '"new terminal"', '"grand opening"', '"hangar opening"',
    '"maintenance base"', '"flight path launch"'
]

# Time window for search
WINDOW_DAYS = 30
WINDOW_END = date.today()
WINDOW_START = WINDOW_END - timedelta(days=WINDOW_DAYS)

In [ ]:
def search_edgar_one_phrase(phrase, start_date, end_date, max_pages=2):
    """
    Issues a paginated search request to the SEC EDGAR search index for a specific phrase.

    Queries the EFTS (Electronic Filing Text Search) API for 8-K filings containing
    the provided search term within a specific date window.

    Args:
        phrase (str): The search term or phrase to look for in filings.
        start_date (datetime.date): The beginning of the search time window.
        end_date (datetime.date): The end of the search time window.
        max_pages (int, optional): The maximum number of result pages to fetch.
            Defaults to 2.

    Returns:
        list: A list of 'hit' dictionaries, where each hit represents a unique SEC filing
            matching the search criteria.
    """
    all_hits = []
    for page in range(max_pages):
        params = {
            "q": phrase,
            "dateRange": "custom",
            "startdt": start_date.isoformat(),
            "enddt": end_date.isoformat(),
            "forms": "8-K",
            "from": page * 100,
        }
        response = requests.get(EDGAR_SEARCH_URL, params=params, headers=REQUEST_HEADERS, timeout=30)
        response.raise_for_status()
        data = response.json()
        hits = data.get("hits", {}).get("hits", [])
        all_hits.extend(hits)
        total = data.get("hits", {}).get("total", {}).get("value", 0)
        if (page + 1) * 100 >= total:
            break
        time.sleep(0.15)
    return all_hits

def fetch_exhibit_text(hit, max_chars=8000):
    """
    Downloads an HTML exhibit from the SEC server and converts it to plain text.

    Constructs the official SEC Archives URL using the CIK and accession number
    provided in the search hit, retrieves the HTML content, and strips tags
    to extract the primary text.

    Args:
        hit (dict): A search hit dictionary from the EDGAR search index.
        max_chars (int, optional): The maximum number of characters to return
            to avoid token limit issues. Defaults to 8000.

    Returns:
        tuple: A tuple containing (text, url) where:
            - text (str): The extracted plain text of the filing.
            - url (str): The direct URL to the SEC filing exhibit.
    """
    accession_full, filename = hit["_id"].split(":")
    accession_no_dashes = accession_full.replace("-", "")
    cik = hit["_source"]["ciks"][0].lstrip("0")
    url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession_no_dashes}/{filename}"

    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)

    return text[:max_chars] if len(text) > max_chars else text, url

def get_industry_candidates(phrases, industry_label):
    """
    Processes a list of search phrases to identify and organize potential filings by industry.

    Iterates through multiple search terms, removes duplicate filings based on
    accession numbers, and aggregates the results into a structured list of
    candidate filings with metadata.

    Args:
        phrases (list): A list of strings containing the keywords or phrases to search.
        industry_label (str): A descriptive name for the industry (e.g., "Financial Services").

    Returns:
        list: A list of dictionaries, each containing company names, tickers,
            filing dates, industry labels, source URLs, and the filing text.
    """
    unique_hits = {}
    print(f"Searching for {industry_label}...")
    for phrase in phrases:
        hits = search_edgar_one_phrase(phrase, WINDOW_START, WINDOW_END)
        for hit in hits:
            unique_hits[hit["_id"]] = hit

    results = []
    for hit_id, hit in unique_hits.items():
        try:
            text, url = fetch_exhibit_text(hit)
            src = hit["_source"]
            results.append({
                "company": (src.get("display_names") or ["(unknown)"])[0],
                "ticker": (src.get("tickers") or [None])[0],
                "file_date": src.get("file_date"),
                "industry": industry_label,
                "url": url,
                "text": text
            })
            time.sleep(0.15)
        except Exception as e:
            print(f"Error fetching {hit_id}: {e}")
    return results

In [ ]:
fin_candidates = get_industry_candidates(FINANCIAL_SERVICES_PHRASES, "Financial Services")

Searching for Financial Services...


In [ ]:
travel_candidates = get_industry_candidates(TRAVEL_HOSPITALITY_PHRASES, "Travel & Hospitality")

Searching for Travel & Hospitality...


In [ ]:
all_candidates = fin_candidates + travel_candidates
print(f"Total candidates retrieved: {len(all_candidates)}")

Total candidates retrieved: 349


In [ ]:
EXTRACTION_SYSTEM_PROMPT = """You are an analyst identifying corporate facility changes (openings, closings, consolidations, or expansions).
Return ONLY a JSON object with:
- is_location_event: boolean (True if a specific facility change is announced)
- event_type: "opening", "closing", "relocation", "expansion", or "other"
- city: string name
- state: two-letter US code
- summary: one sentence summary (<25 words)

Be strict. Ignore mentions in boilerplate or signatures."""

def extract_with_claude(filing):
  """
    Uses the Claude LLM to analyze filing text and extract structured facility event data.

    Sends the filing content to the Anthropic API with a system prompt designed
    to identify corporate location changes. The model determines if an event
    occurred and classifies the city, state, and event type.

    Args:
        filing (dict): A dictionary containing 'company', 'industry', and 'text' keys.

    Returns:
        dict: The original filing dictionary updated with extracted JSON fields:
            - is_location_event (bool)
            - event_type (str)
            - city (str)
            - state (str)
            - summary (str)
            - input_tokens/output_tokens (int)
    """
    user_message = f"Filing from {filing['company']} ({filing['industry']}):\n\n{filing['text']}"
    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=300,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}],
    )
    raw = response.content[0].text.strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()

    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"is_location_event": False, "_parse_error": raw[:200]}

    record = {**filing, **parsed}
    record["input_tokens"] = response.usage.input_tokens
    record["output_tokens"] = response.usage.output_tokens
    return record

def process_all_extractions(candidates):
  """
    Orchestrates the batch analysis of multiple filing candidates using Claude.

    Loops through a list of candidate filings, calls the extraction logic for each,
    and filters out filings that do not contain a confirmed facility event.
    Includes basic progress logging and error handling.

    Args:
        candidates (list): A list of filing dictionaries to be analyzed.

    Returns:
        list: A filtered list of dictionaries representing confirmed corporate
            location events.
    """
    events = []
    print(f"Analyzing {len(candidates)} filings with Claude...")
    for i, filing in enumerate(candidates):
        try:
            result = extract_with_claude(filing)
            if result.get("is_location_event"):
                events.append(result)
        except Exception as e:
            print(f"Error analyzing filing {i}: {e}")

        # Progress indicator
        if (i + 1) % 25 == 0:
            print(f"Processed {i + 1} of {len(candidates)}...")

    return events

# Execute the extraction
location_events = process_all_extractions(all_candidates)
print(f"Found {len(location_events)} genuine location events.")

Analyzing 349 filings with Claude...
Processed 25 of 349...
Processed 50 of 349...
Processed 75 of 349...
Processed 100 of 349...
Processed 125 of 349...
Processed 150 of 349...
Processed 175 of 349...
Processed 200 of 349...
Processed 225 of 349...
Processed 250 of 349...
Processed 275 of 349...
Processed 300 of 349...
Processed 325 of 349...
Found 84 genuine location events.


In [20]:
def geocode_location(city, state):
    """
    Converts a US city and state into geographic coordinates using Nominatim.

    Issues a request to the OpenStreetMap Nominatim API to retrieve latitude
    and longitude. Implements a rate-limit pause to comply with the
    Nominatim Usage Policy (1 request per second).

    Args:
        city (str): The name of the city.
        state (str): The two-letter US state code.

    Returns:
        tuple: A tuple of (latitude, longitude) as floats if found; otherwise, None.
    """
    if not city:
        return None

    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    params = {"q": query, "format": "json", "limit": 1, "countrycodes": "us"}

    try:
        response = requests.get(
            NOMINATIM_URL,
            params=params,
            headers=REQUEST_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()

        # Respect Nominatim's usage policy of 1 request per second
        time.sleep(NOMINATIM_PAUSE)

        if not data:
            return None
        return float(data[0]["lat"]), float(data[0]["lon"])
    except Exception as e:
        print(f"Geocoding error for {query}: {e}")
        return None

# Geocode the identified events
final_geocoded_events = []
print(f"Geocoding {len(location_events)} locations...")
for event in location_events:
    coords = geocode_location(event.get("city"), event.get("state"))
    if coords:
        event["lat"], event["lon"] = coords
        final_geocoded_events.append(event)

print(f"Successfully geocoded {len(final_geocoded_events)} events.")

Geocoding 84 locations...
Successfully geocoded 79 events.


In [21]:
EVENT_COLORS = {
    "opening": "green",
    "closing": "red",
    "relocation": "orange",
    "expansion": "blue",
    "other": "gray",
}

# Map industry to FontAwesome icons
INDUSTRY_ICONS = {
    "Financial Services": "university",
    "Travel & Hospitality": "bed"
}

# Calculate center of map based on geocoded points
if final_geocoded_events:
    avg_lat = sum(e["lat"] for e in final_geocoded_events) / len(final_geocoded_events)
    avg_lon = sum(e["lon"] for e in final_geocoded_events) / len(final_geocoded_events)
else:
    avg_lat, avg_lon = 39.8, -98.6 # Default US center

# Create the map
m = folium.Map(
    location=[avg_lat, avg_lon],
    zoom_start=4,
    tiles="CartoDB positron",
)

for event in final_geocoded_events:
    color = EVENT_COLORS.get(event.get("event_type"), "gray")
    icon_type = INDUSTRY_ICONS.get(event["industry"], "info-sign")

    # Build a clean HTML popup
    popup_html = f"""
    <div style="width: 250px; font-family: Arial;">
        <h4 style="margin-bottom: 5px;">{event['company']}</h4>
        <p style="margin: 0;"><b>Ticker:</b> {event.get('ticker', 'N/A')}</p>
        <p style="margin: 0;"><b>Industry:</b> {event['industry']}</p>
        <p style="margin: 0;"><b>Date:</b> {event['file_date']}</p>
        <p style="margin: 5px 0;"><b>Event:</b> <span style="color: {color}; font-weight: bold;">{event['event_type'].title()}</span></p>
        <p style="font-size: 0.9em; color: #555;">{event['summary']}</p>
        <a href="{event['url']}" target="_blank">View SEC Filing</a>
    </div>
    """

    folium.Marker(
        location=[event["lat"], event["lon"]],
        popup=folium.Popup(popup_html, max_width=350),
        tooltip=f"{event['company']} ({event['industry']})",
        icon=folium.Icon(color=color, icon=icon_type, prefix="fa"),
    ).add_to(m)

# Save and display
OUTPUT_PATH = "industry_location_events_map.html"
m.save(OUTPUT_PATH)
print(f"Map saved to {OUTPUT_PATH}")
m

Map saved to industry_location_events_map.html
